<a href="https://colab.research.google.com/github/diegovc1987-boop/Ejercicio-4/blob/main/Ejercicio4_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%%capture
!pip install gradio transformers google-generativeai deep-translator

### Setup and API Key Configuration

We'll import the necessary libraries and configure the Gemini API key. Make sure your `GEMINI_API_KEY` is set in Colab's 'Secrets' panel (the key icon on the left sidebar).

In [14]:
import gradio as gr
from transformers import pipeline
import google.generativeai as genai
from deep_translator import GoogleTranslator
from google.colab import userdata
import os
import asyncio # Added for asynchronous functions

# Configure Gemini API Key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY") # Corrected: Removed os.getenv as userdata.get only takes one argument

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    # Using a faster Gemini model for quicker summarization
    # Updated model name to a valid 'flash' model, e.g., 'gemini-flash-latest'
    gemini_model = genai.GenerativeModel('gemini-flash-latest')
else:
    print("GOOGLE_API_KEY not found. Gemini summarization will not be available.")
    gemini_model = None

# Load HuggingFace sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### Sentiment Analysis Function

This function uses a pre-trained HuggingFace model to determine the sentiment (positive or negative) of the input text.

In [8]:
def analyze_sentiment(text):
    if not text:
        return "Please enter some text."
    if not sentiment_pipeline:
        return "Sentiment analysis model not loaded."
    try:
        # Translate text to English before sentiment analysis
        translator = GoogleTranslator(source='auto', target='en')
        translated_text = translator.translate(text)

        result = sentiment_pipeline(translated_text[:512])[0]  # Truncate to 512 tokens
        sentiment = "POSITIVO 😊" if result["label"] == "POSITIVE" else "NEGATIVO 😞"
        score = round(result["score"], 4)
        return f"Sentiment: {sentiment} (Confidence: {score*100:.2f}%) - Original: '{text}', Translated: '{translated_text}'"
    except Exception as e:
        return f"Error in sentiment analysis: {e}"

sentiment_interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Enter text for sentiment analysis..."),
    outputs="text",
    title="📊 Sentiment Analyzer",
    description="Analyze the sentiment of your text (Positive or Negative)."
)

### Text Summarization Function

This function uses the Gemini model to generate a concise summary of the provided text.

In [15]:
async def summarize_text(text):
    if not text:
        return "Please enter some text."
    if not gemini_model:
        return "Gemini model not configured. Please set GOOGLE_API_KEY in Colab secrets."

    MAX_SUMMARY_LENGTH = 10000 # Define a maximum length for summarization input
    original_text_length = len(text)
    if original_text_length > MAX_SUMMARY_LENGTH:
        print(f"Warning: Input text for summarization is too long ({original_text_length} chars). Truncating to {MAX_SUMMARY_LENGTH} chars.")
        text = text[:MAX_SUMMARY_LENGTH]

    try:
        prompt = f"""Resume the following text in Spanish concisely (max 3 sentences):

{text}"""
        print(f"[Summarizer] Processing text of length: {len(text)} for summarization.") # Diagnostic print
        print("[Summarizer] Calling gemini_model.generate_content with 60s timeout...") # New diagnostic print
        response = await gemini_model.generate_content(prompt, request_options={'timeout': 60}) # Added explicit timeout
        print("[Summarizer] Received response from gemini_model.generate_content.") # New diagnostic print
        return response.text
    except Exception as e:
        return f"Error in summarization: {e}"

summarization_interface = gr.Interface(
    fn=summarize_text,
    inputs=gr.Textbox(lines=7, placeholder="Enter text to summarize..."),
    outputs="text",
    title="📝 Text Summarizer",
    description="Get a concise summary of your text using Gemini."
)


### Translation Function

This function leverages `deep_translator` to translate text to a specified target language.

In [16]:
def translate_text(text, target_lang="es"):
    if not text:
        return "Please enter some text."
    try:
        translator = GoogleTranslator(source='auto', target=target_lang)
        result = translator.translate(text[:5000])  # Limit character count
        return result
    except Exception as e:
        return f"Error in translation: {e}"

translation_interface = gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(lines=5, placeholder="Enter text to translate..."),
        gr.Dropdown(
            choices=["es", "en", "fr", "de", "it", "pt", "ja", "zh-CN"],
            label="Target Language",
            value="es"
        )
    ],
    outputs="text",
    title="🌍 Translator",
    description="Translate text to various languages."
)

### Launch the Gradio Multi-Tool Application

Finally, we combine all three interfaces into a single `gr.TabbedInterface` and launch the application. The public URL will be displayed below after execution.

In [17]:
import asyncio
import time

async def direct_gemini_test_with_timeout():
    print("Starting isolated Gemini API test with strict asyncio.wait_for timeout...")
    test_prompt = "Hello, what is your name?"

    # Use asyncio.wait_for to enforce a strict 10-second timeout
    try:
        start_time = time.time()
        print(f"[{time.time() - start_time:.2f}s] Attempting generate_content for 10 seconds...")
        response = await asyncio.wait_for(gemini_model.generate_content(test_prompt), timeout=10)
        end_time = time.time()
        print(f"[{end_time - start_time:.2f}s] Received response: {response.text}")
    except asyncio.TimeoutError:
        end_time = time.time()
        print(f"[{end_time - start_time:.2f}s] ERROR: Gemini API call timed out after 10 seconds using asyncio.wait_for.")
    except Exception as e:
        end_time = time.time()
        print(f"[{end_time - start_time:.2f}s] ERROR: An unexpected error occurred: {e}")
    print("Finished isolated Gemini API test.")

await direct_gemini_test_with_timeout()

Starting isolated Gemini API test with strict asyncio.wait_for timeout...
[0.00s] Attempting generate_content for 10 seconds...
[2.44s] ERROR: An unexpected error occurred: object GenerateContentResponse can't be used in 'await' expression
Finished isolated Gemini API test.


In [18]:
import google.generativeai as genai

print("Listing available Gemini models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"  Model: {m.name}, Description: {m.description}")

Listing available Gemini models:
  Model: models/gemini-2.5-flash, Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
  Model: models/gemini-2.5-pro, Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro
  Model: models/gemini-2.0-flash, Description: Gemini 2.0 Flash
  Model: models/gemini-2.0-flash-001, Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
  Model: models/gemini-2.0-flash-lite-001, Description: Stable version of Gemini 2.0 Flash-Lite
  Model: models/gemini-2.0-flash-lite, Description: Gemini 2.0 Flash-Lite
  Model: models/gemini-2.5-flash-preview-tts, Description: Gemini 2.5 Flash Preview TTS
  Model: models/gemini-2.5-pro-preview-tts, Description: Gemini 2.5 Pro Preview TTS
  Model: models/gemma-4-26b-a4b-it, Description: Gemma 4 26B A4B IT
  Model: models/gemma-4-31b-it

Please run the above cell and share the output. This will tell us the correct model names to use for `gemini_model = genai.GenerativeModel(...)`.

Please share the complete output of this cell after you run it. This will tell us definitively if the API call itself is able to complete or if it's consistently hanging/timing out.

In [19]:
multi_tool_app = gr.TabbedInterface(
    [sentiment_interface, summarization_interface, translation_interface],
    ["Sentiment Analyzer", "Text Summarizer", "Translator"],
    title="AI Multi-Tool (Gradio)"
)

multi_tool_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://452a230ad6b5a86420.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
